<!-- nav -->
[← 6 · Calls, members and subscripts](06_Calls_Members_And_Subscripts.ipynb) · [Home](../../README.md) · [9 · Constraints as data →](09_Constraints_As_Data.ipynb)

# 7 · Bindings

Tutorial 4 showed a name as a reference: a term that reads `x` finds its value in its scope. Some expressions give a
name its value themselves: `let c = ... in ...`, a lambda's parameters, a quantifier's item, a comprehension's `for`.
Each **binds** a name within part of itself, so the names it binds are not names its scope must give.

Every language writes these differently, and each dialect models them with kinds of its own, but every kind that binds
names is a variation of one, the framework's `let`: it refines `Let`, whose `name`, `value` and `body` it holds first,
so that what it binds, from what and where reads alike in every language
([CAP-279: names an expression binds found alike in every language](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-279.md)).
This notebook builds a `let`, then lambdas, quantifiers and comprehensions, each in the builders' calls beside its
language's text, and asks each what it needs from its scope with `Symbolics.free` (tutorial 4).

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `variable` writes a
name in the dialects whose names are the framework's `variable`, and `translate` translates an expression from one
dialect to another, as tutorial 6 does; `text` prints a term, or `-` for an empty slot, and `needs` lists the names a
term needs. Python's notebook also reads Python's text, with `Python3.Text.FromText`, which only Python can (tutorial
3); here each Python model is built.

In [1]:
import assert from "node:assert/strict";
import * as Translators from "@mbse/expressions/Translators";
import { Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as C from "@mbse/expressions/C/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import { Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects, show } from "./toolkit.js";

const store = dialects();
const [S, Term] = [store.Expressions, store.Expressions.Framework.Term];
const amd64 = store.Instances.Expressions.SystemV.AMD64;
const variable = (name: string) => (d: any) => d.variable((v: any) => v.name(name));

/** `expression`, of the dialect `source`, translated to `target`, written as a term. */
function translate(expression: unknown, source: any, target: any): any {
  return store.transaction(() => Term(Translators.between(source, target).forward(expression)));
}

/** A term's text as `printer` writes it, or `-` for an empty slot. */
function text(printer: (term: any) => string, term: any): string {
  return term == null ? "-" : printer(term);
}

/** The names a term needs from its scope, in order. */
const needs = (term: unknown, dialect: any): string => [...Symbolics.free(term, dialect)].sort().join(", ");

<a id="T07.1"></a>
## 07.1 · let

Patterns' `let c = get(s, 'celsius') in and(ge(c, 2.0), le(c, 8.0))` reads a shipment's temperature once, names it
`c`, and checks it twice. A `let` is a term of the framework's kind `let`, the schema `Expressions.Framework.Let`: its
`name`, the name it binds, an attribute; its `value`, the term whose value the name takes; and its `body`, the term
the name is bound in. Its value and body are the framework's terms, each held in its dialect's branch (`t.patterns`):

In [2]:
const letExpression = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.let((l: any) => l
  .name("c")
  .value((t: any) => t.patterns((d: any) => d.call((c: any) => c.callable("get").arguments([
    (t: any) => t.patterns(variable("s")),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar("celsius"))))]))))
  .body((t: any) => t.patterns((d: any) => d.call((c: any) => c.callable("and").arguments([
    (t: any) => t.patterns((d: any) => d.call((m: any) => m.callable("ge").arguments([
      (t: any) => t.patterns(variable("c")),
      (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(2.0))))]))),
    (t: any) => t.patterns((d: any) => d.call((m: any) => m.callable("le").arguments([
      (t: any) => t.patterns(variable("c")),
      (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(8.0))))])))]))))))));
const letForm = Terms.form(Terms.term(letExpression));
const [value, body] = letForm.arguments;
console.log(show(letExpression));
console.log(letForm.kind, "name:", letForm.attributes.get("name"), " value:", show(value), " body:", show(body));

let c = get(s, 'celsius') in and(ge(c, 2.0), le(c, 8.0))


let name: c  value: get(s, 'celsius')  body: and(ge(c, 2.0), le(c, 8.0))


`c` is bound in the body alone. The body needs `c`, which the `let` gives it; the value needs `s`, which nothing in the
term gives. So the whole term needs `s` from its scope, and `validate` reports `s`, never `c`:

In [3]:
for (const [part, label] of [[value, "value"], [body, "body"], [Terms.term(letExpression), "let"]] as [unknown, string][]) {
  console.log(label.padEnd(5), "needs", needs(part, E.DIALECT));
}
console.log(JSON.stringify(E.DIALECT.validate(letExpression)));
console.log(JSON.stringify(E.DIALECT.validate(letExpression, ["s"])));
assert.equal(needs(letExpression, E.DIALECT), "s");

value needs s


body  needs c


let   needs s


["value: argument 0: variable 's' is not bound"]


[]


<a id="T07.2"></a>
## 07.2 · let in each language

Three languages have a `let` of their own, and each dialect holds it as the framework's `Let` itself, its value and body
that dialect's terms:

- **Python3** has no `let` expression, but a lambda applied at once is one, `(lambda c: ...)(s.celsius)`: Python3's
  `let` is printed so, and Python reads that text back as a `let` (Python's notebook shows it).
- **Excel**'s `LET(c, s.celsius, ...)`, since Excel 365.
- **LaTeX**'s `... \quad \text{where } c = ...`, the way a paper writes it.

In [4]:
const lets: Record<string, [any, any]> = store.transaction(() => ({
  Python3: [Python3, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.let((l: any) => l.name("c")
    .value((t: any) => t.python3((d: any) => d.attribute((a: any) => a.value(variable("s")).attr("celsius"))))
    .body((t: any) => t.python3((d: any) => d.boolop((b: any) => b.op("and").values([
      (v: any) => v.compare((c: any) => c.left(variable("c")).comparisons([(k: any) => k.op(">=").comparator(
        (t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(2.0)).text("2.0")))])),
      (v: any) => v.compare((c: any) => c.left(variable("c")).comparisons([(k: any) => k.op("<=").comparator(
        (t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(8.0)).text("8.0")))]))])))))))],
  Excel: [Excel, S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.let((l: any) => l.name("c")
    .value((t: any) => t.excel((d: any) => d.field((f: any) => f.value(variable("s")).name("celsius"))))
    .body((t: any) => t.excel((d: any) => d.call((c: any) => c.callable("AND").arguments([
      (t: any) => t.excel((d: any) => d.infix((i: any) => i.left(variable("c")).operator(">=")
        .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(2.0)))))),
      (t: any) => t.excel((d: any) => d.infix((i: any) => i.left(variable("c")).operator("<=")
        .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(8.0))))))])))))))],
  LaTeX: [Latex, S.LaTeX.Expressions.Expression((e: any) => e.root((u: any) => u.let((l: any) => l.name("c")
    .value((t: any) => t.latex((d: any) => d.member((m: any) => m.value(variable("s")).name("celsius"))))
    .body((t: any) => t.latex((d: any) => d.binary((b: any) => b.operator("\\land")
      .left((o: any) => o.binary((b: any) => b.left(variable("c")).operator("\\geq")
        .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(2.0))))))
      .right((o: any) => o.binary((b: any) => b.left(variable("c")).operator("\\leq")
        .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(8.0))))))))))))],
}));
for (const [language, [dialect, built]] of Object.entries(lets)) {
  const written = dialect.Text.ToText(built);
  assert.equal(written, dialect.Text.ToText(translate(letExpression, E.DIALECT, dialect.Expressions.DIALECT))); // as the translator writes it
  console.log(language.padEnd(8), Terms.form(Terms.term(built)).kind, `needs ${needs(built, dialect.Expressions.DIALECT)}: ${written}`);
}
assert.deepEqual(Object.values(lets).map(([dialect, built]) => dialect.Text.ToText(built)), [
  "(lambda c: c >= 2.0 and c <= 8.0)(s.celsius)", "=LET(c, s.celsius, AND(c >= 2.0, c <= 8.0))",
  "c \\geq 2.0 \\land c \\leq 8.0 \\quad \\text{where } c = s.\\mathit{celsius}"]);

Python3  let needs s: (lambda c: c >= 2.0 and c <= 8.0)(s.celsius)


Excel    let needs s: =LET(c, s.celsius, AND(c >= 2.0, c <= 8.0))


LaTeX    let needs s: c \geq 2.0 \land c \leq 8.0 \quad \text{where } c = s.\mathit{celsius}


C, SystemVerilog and MATLAB have no expression that binds a name to a value. The translator writes the value where the
body reads the name, so the text means the same, but it is no longer a `let`: the value is read twice, and the tree that
comes back to Patterns is the substituted one, not the `let`:

In [5]:
for (const dialect of [C, SystemVerilog, Matlab] as any[]) {
  const target = dialect.Expressions.DIALECT;
  const there = translate(letExpression, E.DIALECT, target);
  const back = translate(there, target, E.DIALECT);
  console.log(target.name().padEnd(13), Terms.form(there).kind.padEnd(16), dialect.Text.ToText(there));
  console.log("".padEnd(13), `back: ${show(back)}`);
  assert.ok(!Terms.same(back, letExpression));
}

C             binaryexpression s.celsius >= 2.0 && s.celsius <= 8.0


              back: and(ge(get(s, 'celsius'), 2.0), le(get(s, 'celsius'), 8.0))


SystemVerilog binaryexpression s.celsius >= 2.0 && s.celsius <= 8.0


              back: and(ge(get(s, 'celsius'), 2.0), le(get(s, 'celsius'), 8.0))


MATLAB        binary           s.celsius >= 2.0 && s.celsius <= 8.0


              back: and(ge(get(s, 'celsius'), 2.0), le(get(s, 'celsius'), 8.0))


<a id="T07.3"></a>
## 07.3 · A lambda

A lambda binds its parameters in its body: Python's `lambda x: x + k` and TypeScript's `(x) => x + k` each need only
`k` from their scope. Python3's `lambda` and TypeScript5's `arrowfunctionexpression` each refine `Let`: the body is
Let's `body`, and the parameters, which a `let`'s one `name` can't hold, are their own (Python3's `args`, an `arguments`
helper; TypeScript5's `params`, its patterns). A lambda's value is given when it is called, not when it is written, so
Let's `name` and `value` are left unset, and its form gives None in the value's slot (`-` below):

In [6]:
const lambdas: Record<string, [any, any]> = store.transaction(() => ({
  Python3: [Python3, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.lambda((l: any) => l
    .args((a: any) => a.args([(g: any) => g.arg("x")]))
    .body((b: any) => b.binop((o: any) => o.left(variable("x")).op("+").right(variable("k")))))))],
  TypeScript5: [TypeScript5, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.arrowfunctionexpression((l: any) => l
      .params([(p: any) => p.identifier((i: any) => i.name("x"))])
      .body((b: any) => b.binaryexpression((o: any) => o
        .left((t: any) => t.identifier((i: any) => i.name("x"))).operator("+")
        .right((t: any) => t.identifier((i: any) => i.name("k"))))))))],
}));
for (const [language, [dialect, built]] of Object.entries(lambdas)) {
  const form = Terms.form(Terms.term(built));
  console.log(language.padEnd(12), dialect.Text.ToText(built).padEnd(16), form.kind.padEnd(23),
    `name: ${form.attributes.get("name") ?? "-"}, value:`, text(dialect.Text.ToText, form.arguments[0]),
    " needs", needs(built, dialect.Expressions.DIALECT));
}
assert.deepEqual(Object.values(lambdas).map(([dialect, built]) => dialect.Text.ToText(built)), ["lambda x: x + k", "(x) => x + k"]);

Python3      lambda x: x + k  lambda                  name: -, value: -  needs k


TypeScript5  (x) => x + k     arrowfunctionexpression name: -, value: -  needs k


Validation, which reads the same scopes, reports `k` and not `x`, in each language's words:

In [7]:
for (const [language, [dialect, built]] of Object.entries(lambdas)) {
  console.log(language.padEnd(12), JSON.stringify(dialect.Expressions.DIALECT.validate(built)),
    JSON.stringify(dialect.Expressions.DIALECT.validate(built, ["k"])));
}

Python3      ["body: right: variable 'k' is not bound"] []


TypeScript5  ["body: right: identifier 'k' is not bound"] []


<a id="T07.4"></a>
## 07.4 · Quantifiers bind too

Patterns' `all(get(s, 'readings'), c => le(c, 8.0))` asks whether every reading is at most 8.0: a `quantifier` binds its
`name`, `c`, to each item of a collection in turn, evaluates its `body` for each, and combines the results by its
`quantifier`, `all` (case study 11). It refines `Let`: the collection is Let's `value`, the term the name takes its
values from. Unlike `let`'s, its value and body are Patterns' own terms, not the framework's, so they need no `t.patterns`
branch:

In [8]:
const every = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.quantifier((q: any) => q
  .quantifier("all").name("c")
  .value((d: any) => d.call((c: any) => c.callable("get").arguments([
    (t: any) => t.patterns(variable("s")),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar("readings"))))])))
  .body((d: any) => d.call((c: any) => c.callable("le").arguments([
    (t: any) => t.patterns(variable("c")),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(8.0))))])))))));
const everyForm = Terms.form(Terms.term(every));
console.log(show(every));
console.log(everyForm.kind, "name:", everyForm.attributes.get("name"), " quantifier:", everyForm.attributes.get("quantifier"),
  " value:", show(everyForm.arguments[0]), " body:", show(everyForm.arguments[1]));
console.log("needs", needs(every, E.DIALECT));

all(get(s, 'readings'), c => le(c, 8.0))


quantifier name: c  quantifier: all  value: get(s, 'readings')  body: le(c, 8.0)


needs s


Each language that can say "every reading" says it with a binder of its own, and each refines `Let` too, its collection
Let's `value`:

- **Python**: `all(c <= 8.0 for c in s.readings)`, a call of `all` on a generator expression (07.5), whose `for`
  clause binds `c`.
- **Excel**: `AND(MAP(s.readings, LAMBDA(c, c <= 8.0)))`: `MAP` and its `LAMBDA` are one kind, `map`, which binds
  the lambda's parameter to each item.
- **MATLAB**: `all(arrayfun(@(c) c <= 8.0, s.readings))`: `arrayfun` and its anonymous function are one kind,
  `arrayfun`.
- **SystemVerilog**: `s.readings.and(c) with (c <= 8.0)`, an array reduction with an iterator,
  `arraymethodwithexpression`: its value is the method's call, `s.readings.and(c)`, which names the iterator `c`, so
  it leaves Let's `name` unset, and its scopes say what it binds.

Each is the translation of Patterns' quantifier, and translates back to it:

In [9]:
const identifier = (spelling: string) => (n: any) => n.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling(spelling))));
const quantified: Record<string, [any, any]> = store.transaction(() => ({
  Python3: [Python3, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
    .func(variable("all"))
    .args([(a: any) => a.generatorexp((g: any) => g
      .body((b: any) => b.compare((c: any) => c.left(variable("c")).comparisons([(k: any) => k.op("<=").comparator(
        (t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(8.0)).text("8.0")))])))
      .generators([(f: any) => f.target(variable("c"))
        .iter((i: any) => i.attribute((a: any) => a.value(variable("s")).attr("readings"))).ifs([])]))]))))],
  Excel: [Excel, S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
    .callable("AND").arguments([(t: any) => t.excel((d: any) => d.map((m: any) => m.name("c")
      .value((v: any) => v.field((f: any) => f.value(variable("s")).name("readings")))
      .body((b: any) => b.infix((i: any) => i.left(variable("c")).operator("<=")
        .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(8.0))))))))]))))],
  MATLAB: [Matlab, S.MATLAB.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
    .callable("all").arguments([(t: any) => t.matlab((d: any) => d.arrayfun((m: any) => m.name("c")
      .value((v: any) => v.field((f: any) => f.value(variable("s")).name("readings")))
      .body((b: any) => b.binary((i: any) => i.left(variable("c")).operator("<=")
        .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(8.0))))))))]))))],
  SystemVerilog: [SystemVerilog, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
    (u: any) => u.arraymethodwithexpression((m: any) => m
      .value((v: any) => v.callexpression((c: any) => c
        .callee((f: any) => f.memberexpression((m: any) => m
          .value((v: any) => v.memberexpression((r: any) => r.value(identifier("s")).member((i: any) => i.spelling("readings"))))
          .member((i: any) => i.spelling("and"))))
        .arguments([identifier("c")])))
      .body((b: any) => b.binaryexpression((o: any) => o
        .left(identifier("c")).operator("<=").right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(8.0)))))))))],
}));
for (const [language, [dialect, built]] of Object.entries(quantified)) {
  const target = dialect.Expressions.DIALECT;
  const there = translate(every, E.DIALECT, target);
  assert.equal(dialect.Text.ToText(there), dialect.Text.ToText(built)); // as the translator writes it
  assert.ok(Terms.same(translate(built, target, E.DIALECT), every)); // and back
  const binder = [...Terms.walk(Terms.term(built))].find((t) => Terms.isBinder(Terms.kindOf(t)[1].as_schema()));
  console.log(language.padEnd(13), dialect.Text.ToText(built).padEnd(42), Terms.form(binder).kind.padEnd(25), "needs",
    needs(built, target));
}
assert.deepEqual(Object.values(quantified).map(([dialect, built]) => dialect.Text.ToText(built)), [
  "all(c <= 8.0 for c in s.readings)", "=AND(MAP(s.readings, LAMBDA(c, c <= 8.0)))",
  "all(arrayfun(@(c) c <= 8.0, s.readings))", "s.readings.and(c) with (c <= 8.0)"]);

Python3       all(c <= 8.0 for c in s.readings)          generatorexp              needs s


Excel         =AND(MAP(s.readings, LAMBDA(c, c <= 8.0))) map                       needs s


MATLAB        all(arrayfun(@(c) c <= 8.0, s.readings))   arrayfun                  needs s


SystemVerilog s.readings.and(c) with (c <= 8.0)          arraymethodwithexpression needs s


C and LaTeX have no expression that quantifies over a collection: C would need a loop, a statement, and LaTeX's
`\forall` is not modeled yet. The translators say so, rather than guess:

In [10]:
for (const dialect of [C, Latex] as any[]) {
  try {
    translate(every, E.DIALECT, dialect.Expressions.DIALECT);
  } catch (error) {
    console.log(dialect.Expressions.DIALECT.name().padEnd(6), (error as Error).message);
  }
}

C      Patterns quantifier 'all' has no C counterpart


LaTeX  Patterns quantifier 'all' has no LaTeX counterpart


<a id="T07.5"></a>
## 07.5 · Comprehensions and named expressions

Python's `[p.pin for p in ports if p.pin > 0]` is a `listcomp`: its element, `p.pin`, is Let's `body`, and its `for`
clauses are its `generators`, each a `comprehension` helper holding a `target` (a pattern, which tutorial 8 shows),
the `iter` it takes values from and the conditions, `ifs`, that filter them. The target is bound in the conditions, in
later clauses and in the element, not in its own iterable, so the comprehension needs `ports` alone. A comprehension may
have several clauses, so Let's one `name` and `value` are left unset. Python's `set`, `dict` and generator comprehensions
are alike:

In [11]:
const pins = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.listcomp((l: any) => l
  .body((b: any) => b.attribute((a: any) => a.value(variable("p")).attr("pin")))
  .generators([(g: any) => g.target(variable("p")).iter(variable("ports")).ifs([
    (i: any) => i.compare((c: any) => c.left((t: any) => t.attribute((a: any) => a.value(variable("p")).attr("pin")))
      .comparisons([(k: any) => k.op(">").comparator(
        (t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(0n)).text("0").base(10)))]))])])))));
const pinsForm = Terms.form(Terms.term(pins));
console.log(Python3.Text.ToText(pins), "is a", pinsForm.kind);
console.log("name:", pinsForm.attributes.get("name") ?? "-", " value:", text(Python3.Text.ToText, pinsForm.arguments[0]), " body:",
  text(Python3.Text.ToText, pinsForm.arguments[1]));
console.log("needs", needs(pins, Python3.Expressions.DIALECT));
assert.equal(Python3.Text.ToText(pins), "[p.pin for p in ports if p.pin > 0]");
assert.equal(needs(pins, Python3.Expressions.DIALECT), "ports");

[p.pin for p in ports if p.pin > 0] is a listcomp


name: -  value: -  body: p.pin


needs ports


A **named expression**, `(y := f(x))`, binds `y` to its value, `f(x)`, and gives that value. Its `target` is a name
of the enclosing scope, not of the expression's own: `y` is bound for the code after it, which the term does not hold.
So `namedexpr` holds its value as Let's `value`, leaves Let's `body` unset, and needs only what its value needs:

In [12]:
const walrus = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.namedexpr((n: any) => n
  .target(variable("y"))
  .value((v: any) => v.call((c: any) => c.func(variable("f")).args([variable("x")])))))));
const walrusForm = Terms.form(Terms.term(walrus));
console.log(Python3.Text.ToText(walrus), "is a", walrusForm.kind);
console.log("value:", text(Python3.Text.ToText, walrusForm.arguments[0]), " body:", text(Python3.Text.ToText, walrusForm.arguments[1]));
console.log("needs", needs(walrus, Python3.Expressions.DIALECT));
assert.equal(Python3.Text.ToText(walrus), "(y := f(x))");

(y := f(x)) is a namedexpr


value: f(x)  body: -


needs f, x


<a id="T07.6"></a>
## 07.6 · Every binder a variation of let

Every kind that binds names, in every dialect, refines the framework's `Let`, and lists Let's `name`, `value` and `body`
first, where a language's own syntax tree called them something else: Patterns' quantifier held its collection as
`collection`, MATLAB's `arrayfun` and Excel's `map` theirs as `array`, and Python's comprehensions their element as
`elt`; each is now Let's `value` or `body`. A kind leaves unset what it has no name or term for. Each still binds as its
kind says: a quantifier its name to each item, an import what it imports, and a kind with scopes of its own (a lambda's
parameters, a comprehension's targets, an iterator) by them.

Which kinds those are is data, read from the store: each dialect's branch of the framework's `Term` holds its kinds, and
each binder's schema refines `Expressions.Framework.Let`, which a dialect's `let` is itself. A dialect that reads
another's kinds (Python Types reads Python3's) lists them too, so each schema is shown once, under the dialect that
defines it:

In [13]:
const LET = store.registered("Expressions.Framework.Let");
const binders = new Map<string, Map<string, any>>(); // each binder's schema, once, by the dialect that defines it
for (const held of store.registered("Expressions.Framework.Term").branches as any[]) { // each dialect's terms
  for (const kind of held.schema.branches as any[]) {
    if (Terms.isBinder(kind.schema)) {
      assert.ok(kind.schema === LET || kind.schema.refines === LET);
      assert.deepEqual([...kind.schema.properties.keys()].slice(0, 3), ["name", "value", "body"]);
      const owner = kind.schema === LET ? "Framework" : kind.schema.name.split(".").slice(1, -1).join(".");
      if (!binders.has(owner)) binders.set(owner, new Map());
      if (!binders.get(owner)!.has(kind.name)) binders.get(owner)!.set(kind.name, kind.schema);
    }
  }
}
console.log([...binders].map(([owner, kinds]) => `${owner.padEnd(28)} ${[...kinds.keys()].join(" ")}`).join("\n"));
assert.deepEqual([...binders.keys()], ["Framework", "Patterns.Expressions", "Python.Python3", "TypeScript.TypeScript5",
  "MATLAB.Expressions", "Excel.Expressions", "Cpp.Expressions", "SystemVerilog.Expressions", "SystemVerilog.Constraints",
  "TypeScript.Types"]);

Framework                    let
Patterns.Expressions         quantifier
Python.Python3               listcomp setcomp dictcomp generatorexp namedexpr lambda import importfrom
TypeScript.TypeScript5       functionexpression arrowfunctionexpression classexpression tsemptybodyfunctionexpression
MATLAB.Expressions           arrayfun import
Excel.Expressions            map
Cpp.Expressions              lambdaexpression requiresexpression
SystemVerilog.Expressions    arraymethodwithexpression
SystemVerilog.Constraints    foreachconstraint
TypeScript.Types             tsfunctiontype tsconstructortype tsmappedtype


C has none: none of C's expressions binds a name. C++'s `lambdaexpression` and `requiresexpression` bind in a body of
statements that is not part of the term, so they leave all three of Let's properties unset. TypeScript's function and
mapped types, SystemVerilog's `foreach` constraint and the imports are binders too, each in its own dialect.

So the framework reads every binder alike. The binders this notebook built, in six languages, each give their form
Let's `name`, and Let's `value` and `body` first, `-` here where a kind leaves them unset, and `Symbolics.free` reads what
each binds through its kind:

In [14]:
const built: [string, (term: any) => string, any][] = [
  ["Patterns", show, letExpression], ["Patterns", show, every], ["Python3", Python3.Text.ToText, lets.Python3![1]],
  ["Excel", Excel.Text.ToText, lets.Excel![1]], ["LaTeX", Latex.Text.ToText, lets.LaTeX![1]],
  ...Object.entries(lambdas).map(([language, [dialect, expression]]): [string, (term: any) => string, any] =>
    [language, dialect.Text.ToText, expression]),
  ...Object.entries(quantified).map(([language, [dialect, expression]]): [string, (term: any) => string, any] =>
    [language, dialect.Text.ToText, expression]),
  ["Python3", Python3.Text.ToText, pins], ["Python3", Python3.Text.ToText, walrus]];
console.log("".padEnd(13), "kind".padEnd(25), "name".padEnd(5), "value".padEnd(22), "body");
for (const [language, printer, expression] of built) {
  for (const binder of [...Terms.walk(Terms.term(expression))].filter((t) => Terms.isBinder(Terms.kindOf(t)[1].as_schema()))) {
    const form = Terms.form(binder);
    const [value, body] = form.arguments;
    console.log(language.padEnd(13), form.kind.padEnd(25), String(form.attributes.get("name") ?? "-").padEnd(5),
      text(printer, value).padEnd(22), text(printer, body));
  }
}

              kind                      name  value                  body


Patterns      let                       c     get(s, 'celsius')      and(ge(c, 2.0), le(c, 8.0))


Patterns      quantifier                c     get(s, 'readings')     le(c, 8.0)


Python3       let                       c     s.celsius              c >= 2.0 and c <= 8.0


Excel         let                       c     =s.celsius             =AND(c >= 2.0, c <= 8.0)


LaTeX         let                       c     s.\mathit{celsius}     c \geq 2.0 \land c \leq 8.0


... (4 outputs left out)

MATLAB        arrayfun                  c     s.readings             c <= 8.0


SystemVerilog arraymethodwithexpression -     s.readings.and(c)      c <= 8.0


Python3       listcomp                  -     -                      p.pin


Python3       namedexpr                 -     f(x)                   -


## Why it works this way

- **One kind of binding, many syntaxes.** A `let`, a lambda, a quantifier, a comprehension and an array method's
  iterator each give a name a value within part of themselves. Modeled as variations of one kind, each reads alike to
  every tool: what it binds, from what, and where.
- **Each language keeps its own kind.** A dialect still models its language's syntax as that language has it (Python's
  comprehension clauses, SystemVerilog's iterator in its call), so its text prints back as written; refining `Let` adds
  no syntax, only says which of its parts are Let's.
- **Free names say what data a constraint needs.** What a term binds is not what its scope must give, so
  `Symbolics.free` and validation ask the scope only for the rest, in every language alike.
- **A translator that can't keep a binding says so, or keeps its meaning.** Where a language has no `let`, the value is
  written in place of the name; where it has no quantifier, the translator refuses rather than guess.

<!-- nav -->
[← 6 · Calls, members and subscripts](06_Calls_Members_And_Subscripts.ipynb) · [Home](../../README.md) · [9 · Constraints as data →](09_Constraints_As_Data.ipynb)